# Progress tier — partial-credit ladder (D5)

A binary pass/fail on an expert-level bar throws away almost all signal (Trial 1 was 0/1755 yet 8.7% of runs hijacked control flow). This report scores the ordinal ladder **NONE < RAN < CONTROL < MARKER** instead: how far up it does each backend get, and does the reach differ across backends (Friedman omnibus)?

Each cell is collapsed as the **MAX** rung any replicate reached (best-of-N) — the partial-credit analog of the any-success outcome rule, so the top rung (MARKER) agrees with the binary success matrix by construction.

In [ ]:
from analysis import load_runs

# Pin matplotlib (Agg) + RNG seeds for reproducible, headless figures.
plt = load_runs.pin_style(seed=0)

# Load real runs if any exist, else the deterministic synthetic dataset.
records = load_runs.load_all()
backends = load_runs.backend_names(records)
STRATEGY = "zero_shot"  # the held-fixed strategy for the cross-backend test
print(f"{len(records)} runs, backends={backends}")

In [ ]:
import numpy as np

from analysis import stats as st
from analysis.aggregate import aggregate_runs, build_tier_matrix
from harness.record import MAX_TIER_ORDER

summaries = aggregate_runs(records)
# Ordinal progress-tier matrix (NONE=0 < RAN=1 < CONTROL=2 < MARKER=3),
# collapsed per cell as the MAX rung any replicate reached. Friedman is the
# correct repeated-measures omnibus for this ordinal response (same test the
# time/cost notebooks use).
ids, tier_mat = build_tier_matrix(summaries, backends=backends, strategy=STRATEGY)
omnibus = st.friedman(tier_mat)
print(f"paired binaries: {len(ids)}")
print(f"tier omnibus: {omnibus}")

In [ ]:
# Per-backend tier distribution: sum the per-rung histograms (tier_counts)
# across this strategy's cells, then stack each backend's runs as fractions
# NONE -> RAN -> CONTROL -> MARKER. Lead with the distribution, not one bit.
labels = [str(t) for t in MAX_TIER_ORDER]
counts = {b: np.zeros(len(MAX_TIER_ORDER), dtype=float) for b in backends}
for s in summaries:
    if s.strategy == STRATEGY and s.backend_name in counts:
        counts[s.backend_name] += np.asarray(s.tier_counts, dtype=float)

# none, ran, control, marker
colors = ["#bdbdbd", "#6baed6", "#fd8d3c", "#31a354"]
fig, ax = plt.subplots(figsize=(10, 6))
totals = np.array([counts[b].sum() or 1.0 for b in backends])
bottoms = np.zeros(len(backends))
for i, label in enumerate(labels):
    heights = np.array([counts[b][i] for b in backends]) / totals
    ax.bar(backends, heights, bottom=bottoms, label=label, color=colors[i])
    bottoms += heights
ax.set_ylabel("fraction of runs")
ax.set_title(f"progress-tier distribution by backend (Friedman p={omnibus.p_value:.3g})")
ax.legend(title="max tier", loc="upper right")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
load_runs.save_figure(fig, "08_progress_tier", "tier_distribution")
fig

In [ ]:
from IPython.display import Markdown

from harness.record import MaxTier, max_tier_rank


def _share_at_or_above(backend, min_tier):
    """Fraction of a backend's runs that reached at least `min_tier`."""
    col = counts[backend]
    total = col.sum() or 1.0
    lo = max_tier_rank(min_tier)
    reached = sum(col[i] for i, t in enumerate(MAX_TIER_ORDER) if max_tier_rank(t) >= lo)
    return reached / total


rows = [
    f"- **{b}**: MARKER {100 * _share_at_or_above(b, MaxTier.MARKER):.0f}% "
    f"- CONTROL+ {100 * _share_at_or_above(b, MaxTier.CONTROL):.0f}% "
    f"- RAN+ {100 * _share_at_or_above(b, MaxTier.RAN):.0f}%"
    for b in backends
]
verdict = "reject H0" if omnibus.p_value < 0.05 else "fail to reject H0"
Markdown(
    "### Summary - progress-tier omnibus (Friedman, ordinal)\n\n"
    f"chi2 = {omnibus.statistic:.3f}, df = {omnibus.df}, "
    f"p = {omnibus.p_value:.4g}, Kendall's W = {omnibus.kendalls_w:.3f} "
    f"({verdict}); n = {len(ids)} paired binaries.\n\n"
    "Per-backend tier reach (share of runs):\n\n" + "\n".join(rows)
)